# Multilingual Health Q&A — Hybrid Retrieval + Fine-Tuned mT5

**Challenge:** Multilingual Health Question Answering in Low-Resource African Languages

**Task:** Given a health question in one of 8 language/country subsets (English variants
from Ethiopia/Ghana/Kenya/Uganda, plus Akan, Amharic, Luganda, Swahili), generate a
fluent and accurate answer in the **same language**.

**Evaluation metrics** (all computed from one generated string per row):
- `TargetRLF1` — ROUGE-L F1
- `TargetR1F1` — ROUGE-1 F1
- `TargetLLM`  — LLM-as-a-Judge score

## Approach in this notebook

EDA on the released files showed two things worth exploiting:

1. **Heavy answer templating** — only ~18k unique answers out of ~29.8k training rows;
   many answers (canonical STI/contraception explainers etc.) are reused 60+ times for
   differently-phrased questions. A **TF-IDF nearest-neighbour retrieval** baseline is
   therefore a strong, cheap source of high-ROUGE, guaranteed-fluent answers whenever a
   test question closely matches something already in the training set.
2. **Severe subset imbalance** — `Eng_Uga` has 7,624 training rows vs. `Amh_Eth`'s 1,845,
   and the test set mirrors that skew (`Eng_Uga`, `Aka_Gha`, `Eng_Gha` alone are >60% of
   test rows). We oversample the smaller subsets when fine-tuning so the generator
   doesn't collapse toward majority-subset style.

The pipeline is a **hybrid**:

- Retrieval always runs (fast, no GPU needed) and gives a similarity score per prediction.
- A multilingual seq2seq model (**mT5-base** by default) is fine-tuned on all 8 subsets
  jointly, with the target language named in the prompt, to handle novel/paraphrased
  questions the retrieval step can't match well.
- Per-subset similarity thresholds (picked on the validation set) decide, row by row,
  whether to trust the retrieved answer or fall back to the generated one.

> **Compute note:** fine-tuning mT5-base needs a real GPU (this was written to run on an
> AWS GPU instance — a T4/A10/V100-class card with ≥16GB VRAM is comfortable). If you're
> on something smaller, drop `MODEL_NAME` to `google/mt5-small` and lower `FINETUNE_BATCH_SIZE`.

**Files used directly** (no `SampleSubmission.csv` is provided in this repo — the
submission is built from scratch from the test IDs):
- `Training set.csv`, `Validation set.csv`, `Test set.csv` — columns `ID, input, output, subset`
  (test has no `output`).

## 1 — Install and Import Packages

In [ ]:
# Install required packages
!pip install -q scikit-learn pandas numpy rouge-score
!pip install -q transformers sentencepiece accelerate torch datasets

print('Packages installed')


In [ ]:
import re
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', None)

print('Imports complete')


## 2 — Set File Paths

Fixed to match the files actually present in this repo (the original starter template
referenced `Train.csv` / `Test.csv` / `Val.csv` / `SampleSubmission.csv`, none of which
exist here).

In [ ]:
# Project Root & Directory Setup
BASE_DIR = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
DATA_RAW_DIR = BASE_DIR / 'data' / 'raw'
DATA_PROC_DIR = BASE_DIR / 'data' / 'processed'
MODELS_DIR = BASE_DIR / 'models' / 'checkpoints'
SUBMISSIONS_DIR = BASE_DIR / 'submissions'

for d in [DATA_RAW_DIR, DATA_PROC_DIR, MODELS_DIR, SUBMISSIONS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_RAW_DIR / 'Training set.csv'
VAL_PATH   = DATA_RAW_DIR / 'Validation set.csv'
TEST_PATH  = DATA_RAW_DIR / 'Test set.csv'

# Output paths
OUTPUT_TFIDF  = SUBMISSIONS_DIR / 'submission_tfidf_baseline.csv'
OUTPUT_LLM    = SUBMISSIONS_DIR / 'submission_finetuned_llm.csv'
OUTPUT_HYBRID = SUBMISSIONS_DIR / 'submission_hybrid.csv'

for path in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    status = 'OK' if path.exists() else 'MISSING'
    print(f'[{status}] {path}')


## 3 — Load and Preview the Data

In [ ]:
train = pd.read_csv(TRAIN_PATH)
val   = pd.read_csv(VAL_PATH)
test  = pd.read_csv(TEST_PATH)

print(f'Train shape : {train.shape}')
print(f'Val shape   : {val.shape}')
print(f'Test shape  : {test.shape}')
print()
print('Train columns:', train.columns.tolist())
print('Val columns  :', val.columns.tolist())
print('Test columns :', test.columns.tolist())

display(train.head(3))
display(test.head(3))


In [ ]:
print('Subset distribution — train / val / test (% of split)')
dist = pd.DataFrame({
    'train_n': train['subset'].value_counts(),
    'val_n'  : val['subset'].value_counts(),
    'test_n' : test['subset'].value_counts(),
})
dist['test_pct'] = (dist['test_n'] / dist['test_n'].sum() * 100).round(1)
display(dist.sort_values('test_n', ascending=False))

print()
print('Answer duplication in train (evidence the retrieval baseline should work well):')
n_unique_answers = train['output'].astype(str).str.strip().nunique()
print(f'  {n_unique_answers:,} unique answers out of {len(train):,} rows '
      f'({n_unique_answers / len(train):.1%} unique)')


## 4 — Column and Language Setup

In [ ]:
ID_COL       = 'ID'
QUESTION_COL = 'input'
ANSWER_COL   = 'output'
LANG_COL     = 'subset'

# The `subset` column encodes language and country as "<LangCode>_<CountryCode>".
# Only the language prefix matters for the prompt; the country suffix does not
# change the language the model should answer in.
SUBSET_TO_LANGUAGE = {
    'Eng': 'English',
    'Aka': 'Akan',
    'Lug': 'Luganda',
    'Swa': 'Swahili',
    'Amh': 'Amharic',
}

def subset_to_language_name(subset_code: str) -> str:
    """Extract the full language name from a subset code such as 'Amh_Eth'."""
    if not subset_code or not isinstance(subset_code, str):
        return 'English'
    lang_prefix = subset_code.split('_')[0]
    return SUBSET_TO_LANGUAGE.get(lang_prefix, subset_code)

print('Language mapping:')
for code_, name in SUBSET_TO_LANGUAGE.items():
    print(f'  {code_}_* -> {name}')


## 5 — Text Cleaning

In [ ]:
def clean_text(x):
    """Strip whitespace and handle null values."""
    if pd.isna(x):
        return ''
    return str(x).strip()

train[QUESTION_COL] = train[QUESTION_COL].map(clean_text)
train[ANSWER_COL]   = train[ANSWER_COL].map(clean_text)
val[QUESTION_COL]   = val[QUESTION_COL].map(clean_text)
val[ANSWER_COL]     = val[ANSWER_COL].map(clean_text)
test[QUESTION_COL]  = test[QUESTION_COL].map(clean_text)

train = train[(train[QUESTION_COL] != '') & (train[ANSWER_COL] != '')].reset_index(drop=True)
val   = val[(val[QUESTION_COL] != '') & (val[ANSWER_COL] != '')].reset_index(drop=True)
test  = test[test[QUESTION_COL] != ''].reset_index(drop=True)

print(f'Cleaned train shape : {train.shape}')
print(f'Cleaned val shape   : {val.shape}')
print(f'Cleaned test shape  : {test.shape}')


## 6 — Evaluation Utilities

ROUGE-1 and ROUGE-L scoring using whitespace tokenisation, matching the scoring notes
for this challenge and staying safe for non-Latin scripts (Ge'ez, etc.).

In [ ]:
from rouge_score import rouge_scorer

class WhitespaceTokenizer:
    """Whitespace tokeniser — language-agnostic and safe for African scripts."""
    def tokenize(self, text):
        if text is None:
            return []
        return str(text).strip().split()

def compute_rouge(predictions, references):
    """Mean ROUGE-1 and ROUGE-L F1 over a list of predictions/references."""
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'],
                                       tokenizer=WhitespaceTokenizer(),
                                       use_stemmer=False)
    r1_scores, rl_scores = [], []
    for pred, ref in zip(predictions, references):
        score = scorer.score(str(ref), str(pred))
        r1_scores.append(score['rouge1'].fmeasure)
        rl_scores.append(score['rougeL'].fmeasure)
    return {
        'rouge1_f1': float(np.mean(r1_scores)) if r1_scores else 0.0,
        'rougeL_f1': float(np.mean(rl_scores)) if rl_scores else 0.0,
    }

def compute_rouge_per_example(predictions, references):
    """Per-example rougeL F1 — used for threshold grid search."""
    scorer = rouge_scorer.RougeScorer(['rougeL'], tokenizer=WhitespaceTokenizer(), use_stemmer=False)
    return [scorer.score(str(ref), str(pred))['rougeL'].fmeasure for pred, ref in zip(predictions, references)]

def compute_rouge_by_language(predictions, references, languages):
    """Compute ROUGE scores broken down by language/subset."""
    results = {}
    lang_arr = np.array(languages)
    for lang in np.unique(lang_arr):
        mask = lang_arr == lang
        preds_l = [p for p, m in zip(predictions, mask) if m]
        refs_l  = [r for r, m in zip(references, mask) if m]
        results[lang] = compute_rouge(preds_l, refs_l)
    return pd.DataFrame(results).T

print('ROUGE scorer loaded')


## 7 — Baseline: TF-IDF Retrieval

For each test question, find the most similar training question (within the same
`subset`, falling back to a global model for unseen subsets) using character n-gram
TF-IDF, and return its answer.

**Why character n-grams?** They work across scripts (Latin, Ge'ez, etc.) without
language-specific tokenisation, and are cheap enough to run with no GPU.

In [ ]:
class TfidfRetrievalAnswerer:
    """TF-IDF nearest-neighbour retrieval baseline.

    Fits one global model plus one model per group (subset) so that the nearest
    neighbour for a given question is always searched for within the same
    language/country subset, falling back to the global model for unseen groups.
    """

    def __init__(self, question_col, answer_col, group_col=None,
                 ngram_range=(3, 5), max_features=200_000):
        self.question_col = question_col
        self.answer_col = answer_col
        self.group_col = group_col
        self.ngram_range = ngram_range
        self.max_features = max_features
        self.models = {}
        self.global_model = None

    def _fit_single(self, df):
        vectorizer = TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=self.ngram_range,
            min_df=1,
            max_features=self.max_features,
            lowercase=False,  # preserve case for non-Latin scripts
        )
        questions = df[self.question_col].fillna('').astype(str).tolist()
        answers = df[self.answer_col].fillna('').astype(str).tolist()

        X = vectorizer.fit_transform(questions)
        nn = NearestNeighbors(n_neighbors=1, metric='cosine')
        nn.fit(X)

        return {
            'vectorizer': vectorizer,
            'nn': nn,
            'answers': np.array(answers, dtype=object),
            'questions': np.array(questions, dtype=object),
        }

    def fit(self, df):
        self.global_model = self._fit_single(df)
        self.models = {}
        if self.group_col and self.group_col in df.columns:
            for group, sub in df.groupby(self.group_col):
                if len(sub) >= 2:
                    self.models[group] = self._fit_single(sub)
        return self

    def _predict_single(self, model, questions):
        if len(questions) == 0:
            return np.array([], dtype=object), np.array([]), np.array([], dtype=object)
        X = model['vectorizer'].transform(questions)
        dist, idx = model['nn'].kneighbors(X, n_neighbors=1)
        sims = 1 - dist[:, 0]
        preds = model['answers'][idx[:, 0]]
        matched_q = model['questions'][idx[:, 0]]
        return preds, sims, matched_q

    def predict(self, df, question_col=None, group_col=None):
        question_col = question_col or self.question_col
        group_col = group_col or self.group_col

        questions_all = df[question_col].fillna('').astype(str).tolist()
        n = len(df)
        preds = np.empty(n, dtype=object)
        sims = np.zeros(n)
        matched = np.empty(n, dtype=object)

        if group_col and group_col in df.columns:
            groups = df[group_col].fillna('__none__').astype(str).values
        else:
            groups = np.array(['__none__'] * n)

        idx_arr = np.arange(n)
        for group in np.unique(groups):
            mask = groups == group
            sub_questions = [q for q, m in zip(questions_all, mask) if m]
            model = self.models.get(group, self.global_model)
            p, s, mq = self._predict_single(model, sub_questions)
            sub_idx = idx_arr[mask]
            for j, i in enumerate(sub_idx):
                preds[i] = p[j]
                sims[i] = s[j]
                matched[i] = mq[j]

        return preds.tolist(), sims.tolist(), matched.tolist()


GROUP_COL = LANG_COL
print('Group column:', GROUP_COL)


In [ ]:
print('Fitting TF-IDF retrieval baseline on the training partition...')

retriever = TfidfRetrievalAnswerer(
    question_col=QUESTION_COL,
    answer_col=ANSWER_COL,
    group_col=GROUP_COL,
).fit(train)

val_retr_pred, val_retr_sim, val_retr_match = retriever.predict(val, question_col=QUESTION_COL, group_col=GROUP_COL)

metrics_tfidf = compute_rouge(val_retr_pred, val[ANSWER_COL].tolist())
print(f'TF-IDF baseline — validation ROUGE-1 F1: {metrics_tfidf["rouge1_f1"]:.4f}, '
      f'ROUGE-L F1: {metrics_tfidf["rougeL_f1"]:.4f}')

print('\nPer-language ROUGE:')
display(compute_rouge_by_language(val_retr_pred, val[ANSWER_COL].tolist(), val[LANG_COL].tolist()).round(4))

preview = val[[ID_COL, QUESTION_COL, ANSWER_COL]].head(5).copy()
preview['retrieved_answer'] = val_retr_pred[:5]
preview['similarity'] = [f'{s:.3f}' for s in val_retr_sim[:5]]
display(preview)


In [ ]:
print('Generating TF-IDF predictions for the test set...')

test_retr_pred, test_retr_sim, test_retr_match = retriever.predict(test, question_col=QUESTION_COL, group_col=GROUP_COL)

print(f'Generated {len(test_retr_pred)} retrieval predictions')
print(f'Mean similarity: {np.mean(test_retr_sim):.3f}')


## 8 — Fine-Tune a Multilingual Generator (mT5)

Handles questions the retrieval step can't match well against training data.

`google/mt5-base` is the default: solid multilingual coverage (Swahili, Amharic and
English are in its 101 pretrained languages) and a standard, well-supported
`Seq2SeqTrainer` fine-tuning path. Akan and Luganda aren't among mT5's pretrained
languages, but its SentencePiece tokenizer has byte-level fallback for unseen scripts,
and ~3-4k fine-tuning examples per subset is generally enough for it to pick up the
vocabulary and phrasing patterns.

> `facebook/nllb-200-distilled-600M` has native Akan/Twi and Luganda coverage and is
> worth trying as a follow-up, but fine-tuning it correctly for *many* target
> languages in one run requires grouping batches by target language so each batch gets
> the right decoder-start language token — more moving parts than the default path here.
> Swap `MODEL_NAME` and revisit `build_prompt` / generation calls if you go that route.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = 'google/mt5-base'      # drop to 'google/mt5-small' on limited VRAM

MAX_INPUT_LENGTH  = 256
MAX_OUTPUT_LENGTH = 512
BATCH_SIZE_LLM    = 8
NUM_BEAMS         = 4

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'Model  : {MODEL_NAME}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_llm = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
model_llm = model_llm.to(DEVICE)

print(f'{MODEL_NAME} loaded on {DEVICE}')
print(f'Parameters: {sum(p.numel() for p in model_llm.parameters()) / 1e6:.0f}M')


In [ ]:
def build_prompt(question: str, subset_code: str = None) -> str:
    """Prefix the question with a task + target-language description."""
    if subset_code:
        lang_name = subset_to_language_name(subset_code)
        return f'Answer this health question in {lang_name}: {question}'.strip()
    return str(question).strip()


@torch.no_grad()
def generate_answers_batch(questions, subset_codes=None, batch_size=BATCH_SIZE_LLM):
    """Generate answers for a list of questions with the currently loaded model."""
    if subset_codes is None:
        subset_codes = [None] * len(questions)

    model_llm.eval()
    all_answers = []
    n_batches = (len(questions) + batch_size - 1) // batch_size

    for b in range(n_batches):
        start, end = b * batch_size, min((b + 1) * batch_size, len(questions))
        batch_prompts = [build_prompt(q, s) for q, s in zip(questions[start:end], subset_codes[start:end])]

        inputs = tokenizer(batch_prompts, return_tensors='pt', padding=True,
                            truncation=True, max_length=MAX_INPUT_LENGTH).to(DEVICE)

        output_ids = model_llm.generate(
            **inputs,
            max_length=MAX_OUTPUT_LENGTH,
            num_beams=NUM_BEAMS,
            early_stopping=True,
        )
        decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        decoded = [re.sub(r'<extra_id_\d+>', '', d).strip() for d in decoded]
        all_answers.extend(decoded)

    return all_answers


print('Sanity check on 3 validation examples (zero-shot, before fine-tuning)...')
sample = val.head(3)
gen_sample = generate_answers_batch(sample[QUESTION_COL].tolist(), sample[LANG_COL].tolist(), batch_size=3)
for i, (q, ref, gen) in enumerate(zip(sample[QUESTION_COL], sample[ANSWER_COL], gen_sample)):
    print(f'\n[{i+1}] {sample[LANG_COL].iloc[i]}')
    print(f'  Q   : {q[:120]}')
    print(f'  Ref : {ref[:120]}')
    print(f'  Gen : {gen[:120]}')


### 8.1 — Build the Fine-Tuning Dataset (with oversampling of low-resource subsets)

Training rows are duplicated (sampling with replacement) for subsets below the median
subset size, so the model sees a more balanced mix of languages per epoch instead of
mostly learning `Eng_Uga`/`Eng_Gha`/`Aka_Gha` style answers.

In [ ]:
from datasets import Dataset

def balance_by_subset(df, subset_col=LANG_COL, seed=SEED):
    counts = df[subset_col].value_counts()
    target = int(counts.median())
    print(f'Balancing target (median subset size): {target}')

    parts = []
    for subset, sub_df in df.groupby(subset_col):
        if len(sub_df) < target:
            extra = sub_df.sample(n=target - len(sub_df), replace=True, random_state=seed)
            sub_df = pd.concat([sub_df, extra], ignore_index=True)
        parts.append(sub_df)
        print(f'  {subset:10s}: {len(df[df[subset_col] == subset]):5d} -> {len(sub_df):5d}')

    return pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)


train_balanced = balance_by_subset(train)
print(f'\nBalanced training rows: {len(train_balanced):,} (from {len(train):,})')


In [ ]:
FINETUNE_MAX_INPUT_LEN  = MAX_INPUT_LENGTH
FINETUNE_MAX_TARGET_LEN = MAX_OUTPUT_LENGTH

def make_hf_dataset(df, question_col, answer_col, lang_col):
    """Build a tokenized HF Dataset with prompt-formatted inputs.

    Uses build_prompt() so training inputs match inference inputs exactly.
    """
    prompts = [build_prompt(q, s) for q, s in zip(df[question_col], df[lang_col])]
    answers = df[answer_col].astype(str).tolist()
    raw_ds = Dataset.from_dict({'prompt': prompts, 'answer': answers})

    def preprocess(examples):
        model_inputs = tokenizer(examples['prompt'], max_length=FINETUNE_MAX_INPUT_LEN,
                                  truncation=True, padding=False)
        labels = tokenizer(text_target=examples['answer'], max_length=FINETUNE_MAX_TARGET_LEN,
                            truncation=True, padding=False)
        model_inputs['labels'] = labels['input_ids']
        return model_inputs

    return raw_ds.map(preprocess, batched=True, remove_columns=['prompt', 'answer'])


# Small held-out slice of the (already balanced) training data to monitor overfitting.
FINETUNE_VAL_SIZE = 0.05
train_ft = train_balanced.sample(frac=1 - FINETUNE_VAL_SIZE, random_state=SEED)
eval_ft = train_balanced.drop(train_ft.index)

train_dataset = make_hf_dataset(train_ft, QUESTION_COL, ANSWER_COL, LANG_COL)
eval_dataset = make_hf_dataset(eval_ft, QUESTION_COL, ANSWER_COL, LANG_COL)

print(f'Train dataset: {len(train_dataset):,} examples')
print(f'Eval dataset : {len(eval_dataset):,} examples')


### 8.2 — Train

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

FINETUNE_OUTPUT_DIR    = './mt5-finetuned-health-qa'
FINETUNE_EPOCHS        = 3
FINETUNE_BATCH_SIZE    = 8      # reduce to 2-4 on smaller GPUs
FINETUNE_LEARNING_RATE = 5e-5

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model_llm,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

training_args = Seq2SeqTrainingArguments(
    output_dir=FINETUNE_OUTPUT_DIR,
    num_train_epochs=FINETUNE_EPOCHS,
    per_device_train_batch_size=FINETUNE_BATCH_SIZE,
    per_device_eval_batch_size=FINETUNE_BATCH_SIZE,
    learning_rate=FINETUNE_LEARNING_RATE,
    predict_with_generate=True,
    bf16=(DEVICE == 'cuda' and torch.cuda.is_bf16_supported()),
    fp16=(DEVICE == 'cuda' and not torch.cuda.is_bf16_supported()),
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    logging_steps=100,
    generation_max_length=FINETUNE_MAX_TARGET_LEN,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model_llm,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

trainer.train()


### 8.3 — Generate Predictions with the Fine-Tuned Model

In [ ]:
model_llm.eval()

print('Generating fine-tuned predictions for the validation set...')
val_gen_pred = generate_answers_batch(val[QUESTION_COL].tolist(), val[LANG_COL].tolist())

metrics_ft = compute_rouge(val_gen_pred, val[ANSWER_COL].tolist())
print(f'\nFine-tuned LLM — validation ROUGE-1 F1: {metrics_ft["rouge1_f1"]:.4f}, '
      f'ROUGE-L F1: {metrics_ft["rougeL_f1"]:.4f}')

print('\nPer-language ROUGE:')
display(compute_rouge_by_language(val_gen_pred, val[ANSWER_COL].tolist(), val[LANG_COL].tolist()).round(4))


In [ ]:
print('Generating fine-tuned predictions for the test set...')
test_gen_pred = generate_answers_batch(test[QUESTION_COL].tolist(), test[LANG_COL].tolist())
print(f'Generated {len(test_gen_pred)} predictions')


## 9 — Combine Baselines: Retrieval + Generation Hybrid

For each subset, grid-search a similarity threshold on the validation set: below the
threshold the retrieved answer is too dissimilar to trust, so fall back to the
fine-tuned model's generated answer; at/above it, the retrieved answer wins (usually
higher ROUGE and guaranteed fluent, since it's a real training answer).

In [ ]:
def hybrid_combine(retr_pred, retr_sim, gen_pred, threshold):
    return [rp if rs >= threshold else gp for rp, rs, gp in zip(retr_pred, retr_sim, gen_pred)]


def pick_best_threshold(retr_pred, retr_sim, gen_pred, references, thresholds=np.arange(0.30, 0.96, 0.05)):
    best_t, best_score = 1.0, -1.0
    for t in thresholds:
        combined = hybrid_combine(retr_pred, retr_sim, gen_pred, t)
        score = compute_rouge(combined, references)['rougeL_f1']
        if score > best_score:
            best_t, best_score = t, score
    return best_t, best_score


val_lang_arr = np.array(val[LANG_COL].tolist())
val_refs = val[ANSWER_COL].tolist()

subset_thresholds = {}
print(f'{"subset":10s} {"best_t":>7s} {"hybrid_rL":>10s} {"retr_rL":>9s} {"gen_rL":>8s}')
for subset in sorted(set(val_lang_arr)):
    mask = val_lang_arr == subset
    rp = [p for p, m in zip(val_retr_pred, mask) if m]
    rs = [s for s, m in zip(val_retr_sim, mask) if m]
    gp = [p for p, m in zip(val_gen_pred, mask) if m]
    refs = [r for r, m in zip(val_refs, mask) if m]

    t, hybrid_score = pick_best_threshold(rp, rs, gp, refs)
    retr_score = compute_rouge(rp, refs)['rougeL_f1']
    gen_score = compute_rouge(gp, refs)['rougeL_f1']
    subset_thresholds[subset] = t

    print(f'{subset:10s} {t:7.2f} {hybrid_score:10.4f} {retr_score:9.4f} {gen_score:8.4f}')

print('\nChosen per-subset thresholds:', subset_thresholds)


In [ ]:
def apply_hybrid(retr_pred, retr_sim, gen_pred, subsets, thresholds_by_subset, default_threshold=0.6):
    out = []
    for rp, rs, gp, s in zip(retr_pred, retr_sim, gen_pred, subsets):
        t = thresholds_by_subset.get(s, default_threshold)
        out.append(rp if rs >= t else gp)
    return out


val_hybrid_pred = apply_hybrid(val_retr_pred, val_retr_sim, val_gen_pred, val[LANG_COL].tolist(), subset_thresholds)
metrics_hybrid = compute_rouge(val_hybrid_pred, val_refs)
print(f'Hybrid — validation ROUGE-1 F1: {metrics_hybrid["rouge1_f1"]:.4f}, '
      f'ROUGE-L F1: {metrics_hybrid["rougeL_f1"]:.4f}')

comparison = pd.DataFrame({
    'Approach': ['TF-IDF retrieval', f'Fine-tuned {MODEL_NAME}', 'Hybrid'],
    'ROUGE-1 F1': [metrics_tfidf['rouge1_f1'], metrics_ft['rouge1_f1'], metrics_hybrid['rouge1_f1']],
    'ROUGE-L F1': [metrics_tfidf['rougeL_f1'], metrics_ft['rougeL_f1'], metrics_hybrid['rougeL_f1']],
})
print()
display(comparison.round(4))


## 10 — Create Submission Files

Each submission needs exactly four columns: `ID`, `TargetRLF1`, `TargetR1F1`, `TargetLLM`,
all three holding the same generated answer.

In [ ]:
def make_submission(ids, predictions, output_path, reference_len=None):
    clean_preds = [re.sub(r'<extra_id_\d+>', '', str(p)).strip() for p in predictions]

    sub = pd.DataFrame({
        'ID': ids,
        'TargetRLF1': clean_preds,
        'TargetR1F1': clean_preds,
        'TargetLLM': clean_preds,
    })[['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']]

    required_cols = ['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']
    assert list(sub.columns) == required_cols, f'Expected columns {required_cols}, got {list(sub.columns)}'
    if reference_len is not None:
        assert len(sub) == reference_len, f'Row count mismatch: {len(sub)} vs {reference_len}'
    assert sub[required_cols[1:]].notna().all().all(), 'Missing values found in submission'
    assert (sub['TargetRLF1'] == sub['TargetR1F1']).all(), 'TargetRLF1 and TargetR1F1 differ'
    assert (sub['TargetRLF1'] == sub['TargetLLM']).all(), 'TargetRLF1 and TargetLLM differ'

    sub.to_csv(output_path, index=False, encoding='utf-8')
    print(f'Saved: {output_path}  shape={sub.shape}')
    display(sub.head(3))
    return sub


test_hybrid_pred = apply_hybrid(test_retr_pred, test_retr_sim, test_gen_pred, test[LANG_COL].tolist(), subset_thresholds)

print('Saving TF-IDF baseline submission...')
sub_tfidf = make_submission(test[ID_COL].values, test_retr_pred, OUTPUT_TFIDF, reference_len=len(test))
print()

print('Saving fine-tuned LLM submission...')
sub_llm = make_submission(test[ID_COL].values, test_gen_pred, OUTPUT_LLM, reference_len=len(test))
print()

print('Saving hybrid submission (recommended)...')
sub_hybrid = make_submission(test[ID_COL].values, test_hybrid_pred, OUTPUT_HYBRID, reference_len=len(test))


## 11 — Advanced Next Steps & Modular Pipeline Implementation

All 5 recommendations from Section 11 have been implemented as modular Python scripts in `src/`:

1. **`facebook/nllb-200-distilled-600M` Generation Model**: Configured in `src/nllb_pipeline.py` with native FLORES-200 language code mappings (`aka_Latn`, `amh_Ethi`, `lug_Latn`, `swh_Latn`, `eng_Latn`) and per-language mini-batch grouping.
2. **Dense & Hybrid RAG Retrieval**: Implemented in `src/retrieval.py` (`HybridRetriever`), combining TF-IDF subword/char n-grams with multilingual dense embeddings (`sentence-transformers/paraphrase-multilingual-mpnet-base-v2`).
3. **Tightened Threshold Grid & Score Calibration**: Implemented in `src/threshold_optimizer.py` (`optimize_per_subset_thresholds`) using `0.01` resolution grid search and Logistic Regression calibrator.
4. **Per-Subset ROUGE Evaluation Callback**: Added `PerSubsetRougeCallback` to track per-language subset ROUGE-1 and ROUGE-L F1 scores during fine-tuning epochs.
5. **TargetLLM Post-Processing**: Added `clean_text_for_target_llm` to normalize spacing, remove special token artifacts, and format predictions for the LLM judge.

In [ ]:
# 11.1 — Test Modular Hybrid Retrieval (TF-IDF + Multilingual Dense Embeddings)
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / 'src')) if Path.cwd().name == 'notebooks' else sys.path.append('src')

from retrieval import HybridRetriever
from threshold_optimizer import optimize_per_subset_thresholds

print('[INFO] Initializing Hybrid Lexical + Dense RAG Retriever...')
hybrid_retriever = HybridRetriever(train, enable_dense=True)

# Run retrieval on validation set
val_retr_preds, val_retr_sims = [], []
for _, row in val.iterrows():
    ans, sim = hybrid_retriever.get_top1(row['input'], row['subset'], exclude_exact=True)
    val_retr_preds.append(ans)
    val_retr_sims.append(sim)

print(f'[SUCCESS] Retrieved {len(val_retr_preds)} hybrid reference answers.')


In [ ]:
# 11.2 — Run Per-Subset Threshold Optimization & Calibration Grid
optimal_thresholds, report_df = optimize_per_subset_thresholds(
    val_df=val,
    retr_preds=val_retr_preds,
    retr_sims=val_retr_sims,
    gen_preds=val_gen_pred,
    grid_step=0.01
)

print('Per-Subset Threshold Optimization Report:')
display(report_df)


In [ ]:
# 11.3 — Execute the Complete NLLB-200 Pipeline Script
# Usage: python src/nllb_pipeline.py --model_name facebook/nllb-200-distilled-600M --use_dense_rag --epochs 3 --batch_size 8
print('[INFO] Command to run full NLLB-200 training and hybrid submission generation:')
print('python src/nllb_pipeline.py --model_name facebook/nllb-200-distilled-600M --use_dense_rag --epochs 3 --batch_size 8')
